# 5.9 Bias-Variance Tradeoff

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.09_bias-variance-tradeoff.ipynb)

Companion code for Section 5.9 of the book. Run the cells in order from top to bottom.

**Example 5.9.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

def make_dataset(n=30, noise=1.0, seed=None):
    '''True function: y = sin(x) + noise'''
    rng = np.random.RandomState(seed)
    X = rng.uniform(0, 2*np.pi, n).reshape(-1, 1)
    y = np.sin(X.squeeze()) + rng.normal(0, noise, n)
    return X, y

def make_model(degree):
    return Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr',   LinearRegression())
    ])

# ── Bias-Variance experiment ──────────────────────────────────────
n_experiments = 100        # Number of independent training datasets
n_train       = 30         # Training set size
degrees       = [1, 3, 9, 15]
noise         = 0.5
x_eval = np.linspace(0, 2*np.pi, 200).reshape(-1, 1)
y_true = np.sin(x_eval.squeeze())

bv_results = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bias-Variance Tradeoff - 100 Models per Degree', fontsize=14, fontweight='bold')

for idx, degree in enumerate(degrees):
    predictions = []   # Store predictions from each experiment

    ax = axes[idx//2, idx%2]

    for exp in range(n_experiments):
        X_tr, y_tr = make_dataset(n_train, noise, seed=exp)
        model = make_model(degree)
        model.fit(X_tr, y_tr)
        pred = model.predict(x_eval)
        predictions.append(pred)

        # Plot each model's curve (light colour)
        if exp < 20:  # Only plot first 20 for clarity
            ax.plot(x_eval, pred, 'b-', alpha=0.1, linewidth=0.8)

    predictions = np.array(predictions)  # Shape: (n_experiments, 200)
    mean_pred  = predictions.mean(axis=0)
    var_pred   = predictions.var(axis=0)

    # Bias² = (mean_prediction - true_value)²
    bias_sq = (mean_pred - y_true)**2

    avg_bias_sq  = bias_sq.mean()
    avg_variance = var_pred.mean()
    total_error  = avg_bias_sq + avg_variance + noise**2

    bv_results[degree] = {
        'bias_sq': avg_bias_sq,
        'variance': avg_variance,
        'total': total_error
    }

    # Plot true function and mean prediction
    ax.plot(x_eval, y_true, 'r-', linewidth=2.5, label='True: sin(x)')
    ax.plot(x_eval, mean_pred, 'k-', linewidth=2.5, label='Mean prediction')
    ax.fill_between(x_eval.squeeze(),
                    mean_pred - 2*np.sqrt(var_pred),
                    mean_pred + 2*np.sqrt(var_pred),
                    alpha=0.2, color='blue', label='±2σ band')

    verdict = (['UNDERFITTING','HIGH BIAS','OPTIMAL','OVERFITTING'][idx])
    ax.set_title(f'Degree {degree} - {verdict}\n'
                f'Bias²={avg_bias_sq:.4f}  Var={avg_variance:.4f}  Total={total_error:.4f}',
                fontsize=10)
    ax.set_ylim(-3, 3); ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bias_variance_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print bias-variance table ─────────────────────────────────────
print(f'{'Degree':>8} {'Bias²':>10} {'Variance':>10} {'Total':>10} {'Verdict'}')
print('-'*55)
for deg, r in bv_results.items():
    b, v, t = r['bias_sq'], r['variance'], r['total']
    verdict = 'OVERFIT' if v > b else ('UNDERFIT' if b > 0.5 else 'BALANCED')
    print(f'{deg:>8} {b:>10.4f} {v:>10.4f} {t:>10.4f}  {verdict}')

# ── Bias-Variance vs Model Complexity curve ───────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
degs  = list(bv_results.keys())
biases = [bv_results[d]['bias_sq'] for d in degs]
vars_  = [bv_results[d]['variance'] for d in degs]
totals = [bv_results[d]['total'] for d in degs]

ax.plot(degs, biases, 'b-o', lw=2.5, ms=8, label='Bias² (underfitting)')
ax.plot(degs, vars_,  'r-o', lw=2.5, ms=8, label='Variance (overfitting)')
ax.plot(degs, totals, 'g-o', lw=2.5, ms=8, label='Total Error', linewidth=3)
ax.axvline(x=3, color='green', linestyle='--', alpha=0.7, label='Optimal degree ≈ 3')

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.set_title('Bias-Variance Tradeoff vs Model Complexity', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
ax.fill_between([0,3], [0,0], [0.4,0.4], alpha=0.1, color='blue', label='Bias-dominated')
ax.fill_between([3,16],[0,0], [0.4,0.4], alpha=0.1, color='red',  label='Variance-dominated')

plt.tight_layout()
plt.savefig('bias_variance_curve.png', dpi=150, bbox_inches='tight')
plt.show()